# Tasmiq AI - Engine Testing

We will install the required dependencies directly into this notebook's environment to ensure they are available, and use a local audio file instead of an online dataset.

In [1]:
!pip install --ignore-installed huggingface-hub transformers torch soundfile librosa numpy


  Using cached huggingface_hub-1.10.1-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-5.5.3-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.11.0-cp312-cp312-win_amd64.whl.metadata (29 kB)
  Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata (16 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.4 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.4.4 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.17.1 which is incompatible.
gradio 5.18.0 requires markupsafe~=2.0, but you have markupsafe 3.0.3 which is incompatible.
mdit-py-plugins 0.3.0 requires markdown-it-py<3.0.0,>=1.0.0, but you have markdown-it-py 4.0.0 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2026.3.0 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you

In [1]:
import os
import sys
import logging
import soundfile as sf
import librosa
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

logging.basicConfig(level=logging.INFO)

def test_engine_local_file(audio_path):
    if not os.path.exists(audio_path):
        logging.error(f"Could not find audio file '{audio_path}'. Please upload one or change the path!")
        return

    logging.info("Loading Wav2Vec2 phonetics model... (This will take a minute to download on first run)")
    processor = Wav2Vec2Processor.from_pretrained("TBOGamer22/wav2vec2-quran-phonetics")
    model = Wav2Vec2ForCTC.from_pretrained("TBOGamer22/wav2vec2-quran-phonetics")
    model.eval()

    logging.info("Reading local audio file...")
    # Load audio using soundfile
    audio_array, sr = sf.read(audio_path)
    
    # Convert stereo to mono if needed
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)

    # Resample to 16kHz as expected by wav2vec2
    if sr != 16000:
        logging.info(f"Resampling from {sr} Hz to 16000 Hz...")
        audio_array = librosa.resample(y=audio_array, orig_sr=sr, target_sr=16000)

    logging.info("Running inference...")
    inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.inference_mode():
        logits = model(inputs.input_values).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    phonetics = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    print("\n--- TEST COMPLETE ---")
    print(f"Audio File Tested: {audio_path}")
    print(f"Phonetics detected: {phonetics}")

# PUT YOUR LOCAL FILE NAME HERE (From your local Quran dataset)
test_file_path = "test_audio.wav" 
test_engine_local_file(test_file_path)

ModuleNotFoundError: No module named 'soundfile'

In [3]:
import sys
!{sys.executable} -m pip install soundfile librosa transformers torch



  Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata (16 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached transformers-5.5.3-py3-none-any.whl.metadata (32 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
  Using cached huggingface_hub-1.10.1-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached soundfile-0.13.1-py2.py3-none-win_am

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\nabil\\AppData\\Local\\Temp\\pip-unpack-q220w18t\\torch-2.11.0-cp310-cp310-win_amd64.whl'
Consider using the `--user` option or check the permissions.

